# Name : Muhammad Danish Zaheer Awan
# REG id : 25280092
## Task 3 - Domain Generalization

## Workflow rules

- Reuse the exact Task 2 source splits and Source-only checkpoint.
- Never retrain ERM inside Task 3.
- Never load or inspect Sketch during training, diagnostics, or selection.
- Select checkpoints only with mean source-validation macro-F1.
- Complete the SAM controlled study and hypotheses before locking decisions.
- Unlock Sketch once for final evaluation and analysis only.

## 0. Imports and project setup

In [ ]:
# Import standard utilities for cleanup, paths, tables, plotting, and PyTorch.
# Locate the repository root so imports work from either notebook location.

import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / 'task3').is_dir() and (path / 'Data').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Run this notebook from inside the PA_1 project.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
# Import the Task 3 source-only PACS protocol and YAML configuration helpers.
# These functions reuse Task 2 data while deliberately avoiding the Sketch folder.

from shared_task_2and3.pacs_protocol import (
    build_task3_source_datasets,
    prepare_task3_source_protocol,
)
from task3.configs.config_loader import (
    create_output_directories,
    get_output_paths,
    load_config,
)

In [ ]:
# Import training and source-only diagnostic functions for the three methods.
# All functions in this cell operate without constructing a target dataset.

from task2.evaluation.metrics import save_json
from task3.evaluation.domain_metrics import evaluate_source_model
from task3.evaluation.sharpness import (
    calculate_local_sharpness,
    create_fixed_validation_batch,
)
from task3.evaluation.source_domain_separability import (
    evaluate_source_domain_separability,
)
from task3.training.train import (
    select_training_device,
    set_random_seed,
    train_or_load_task3_method,
)

In [ ]:
# Import the lock and final Sketch evaluator without executing target access.
# Sketch images are constructed only when the later guarded function is called.

from task3.evaluation.evaluate_sketch import (
    create_experiment_lock,
    run_final_sketch_evaluation,
)

## 1. Runtime and main configurations

In [ ]:
# Define reproducible random generators and automatically prefer the CUDA GPU.
# The same seed is restored before each Task 3 model is initialized.

def configure_runtime(seed=6304):
    set_random_seed(seed)
    return select_training_device()

In [ ]:
# Initialize the runtime and display the exact PyTorch compute environment.
# Training and feature extraction use CUDA while sklearn diagnostics use CPU.

DEVICE = configure_runtime()
print(f'Project root: {project_root}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Define one helper that loads ERM, DAN-DG, and SAM main configurations.
# ERM is marked load-only and points directly to the Task 2 checkpoint.

def load_main_configurations():
    return {
        method_name: load_config(method_name)
        for method_name in ('erm', 'dan_dg', 'sam')
    }

In [ ]:
# Load the main configurations and create Task 3 cache and result directories.
# Display method settings before loading data or checkpoints.

MAIN_CONFIGURATIONS = load_main_configurations()
OUTPUT_PATHS = create_output_directories(MAIN_CONFIGURATIONS['erm'])
display(pd.DataFrame([config['method'] for config in MAIN_CONFIGURATIONS.values()]))

## 2. Reuse the source-only PACS protocol

In [ ]:
# Define source-protocol loading from the PACS data and split prepared by Task 2.
# The implementation validates Photo, Art Painting, and Cartoon without scanning Sketch.

def prepare_task3_sources(configuration):
    paths = get_output_paths(configuration)
    return prepare_task3_source_protocol(
        dataset_directory=paths['dataset_directory'],
        split_file=paths['split_file'],
    )

In [ ]:
# Load only the existing source image root and fixed Task 2 split information.
# Run Task 2 data preparation first if either dependency does not exist.

PACS_SOURCE_ROOT, PACS_PROTOCOL = prepare_task3_sources(
    MAIN_CONFIGURATIONS['erm']
)

In [ ]:
# Define the source training and validation datasets with Task 2 transforms.
# The returned dictionary has no target, adaptation, or Sketch dataset key.

def create_task3_datasets(image_root, protocol, configuration):
    transform_settings = configuration['transforms']
    return build_task3_source_datasets(
        image_root,
        protocol,
        resize_size=transform_settings['resize_size'],
        crop_size=transform_settings['crop_size'],
    )

In [ ]:
# Construct source-only datasets and assert the strict two-key boundary.
# This prevents a target dataset from reaching Task 3 training code.

TASK3_DATASETS = create_task3_datasets(
    PACS_SOURCE_ROOT,
    PACS_PROTOCOL,
    MAIN_CONFIGURATIONS['erm'],
)
assert set(TASK3_DATASETS) == {'source_train', 'source_validation'}

In [ ]:
# Define a compact table for the reused source training and validation splits.
# The counts should match the corresponding Task 2 source-domain counts exactly.

def create_source_size_table(datasets):
    rows = []
    for split_name in ('source_train', 'source_validation'):
        for domain, dataset in datasets[split_name].items():
            rows.append({
                'domain': domain,
                'split': split_name.replace('source_', ''),
                'number_of_examples': len(dataset),
            })
    return pd.DataFrame(rows)

In [ ]:
# Display the source-only dataset sizes before loading the shared ERM baseline.
# No target count or target sample is inspected anywhere in this stage.

display(create_source_size_table(TASK3_DATASETS))

## 3. Common training and memory helper

In [ ]:
# Define one wrapper for load-only ERM and source-only DAN-DG or SAM training.
# Release model tensors afterward so sequential experiments fit on the GPU.

def run_and_release(configuration, datasets, force_retrain=False):
    result = train_or_load_task3_method(
        configuration,
        datasets,
        device=DEVICE,
        force_retrain=force_retrain,
    )
    compact_result = {
        'checkpoint_file': result['checkpoint_file'],
        'history_file': result['history_file'],
        'history': result['history'],
        'selected_epoch': result['checkpoint']['epoch'],
        'selection_value': result['checkpoint']['selection_value'],
        'source_validation': result['checkpoint']['source_validation'],
        'reused_from_task2': result['reused_from_task2'],
    }
    result['model'].to('cpu')
    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return compact_result

In [ ]:
# Choose whether completed DAN-DG and SAM checkpoints should be overwritten.
# This flag never retrains ERM because its Task 2 checkpoint is always load-only.

FORCE_RETRAIN = False
TRAINING_RESULTS = {}

## 4. Shared ERM baseline

In [ ]:
# Load the unchanged source-selected Task 2 Source-only checkpoint as ERM.
# The loader verifies its method and source-validation selection criterion.

TRAINING_RESULTS['erm'] = run_and_release(
    MAIN_CONFIGURATIONS['erm'],
    TASK3_DATASETS,
    force_retrain=False,
)
print(TRAINING_RESULTS['erm']['checkpoint_file'])
print('Reused from Task 2:', TRAINING_RESULTS['erm']['reused_from_task2'])

## 5. DAN-DG

In [ ]:
# Train DAN-DG with the exact Task 2 MMD over all three source pairs.
# Classification and alignment use only Photo, Art Painting, and Cartoon.

TRAINING_RESULTS['dan_dg'] = run_and_release(
    MAIN_CONFIGURATIONS['dan_dg'],
    TASK3_DATASETS,
    force_retrain=FORCE_RETRAIN,
)
display(TRAINING_RESULTS['dan_dg']['history'].tail())

## 6. SAM

In [ ]:
# Train standard non-adaptive SAM with radius 0.05 and wrapped AdamW.
# Both ascent and descent passes reuse the same balanced source images.

TRAINING_RESULTS['sam'] = run_and_release(
    MAIN_CONFIGURATIONS['sam'],
    TASK3_DATASETS,
    force_retrain=FORCE_RETRAIN,
)
display(TRAINING_RESULTS['sam']['history'].tail())

## 7. Source-only training evidence

In [ ]:
# Define plots for classification, MMD, and source-validation performance.
# ERM uses its original Task 2 history while Task 3 methods use new histories.

def plot_training_curves(training_results, output_file):
    figure, axes = plt.subplots(1, 3, figsize=(18, 5))
    for result_name, result in training_results.items():
        history = result['history']
        if history.empty:
            continue
        axes[0].plot(history['epoch'], history['classification_loss'], label=result_name)
        mmd_values = history['mmd_loss'] if 'mmd_loss' in history else history['alignment_loss']
        axes[1].plot(history['epoch'], mmd_values, label=result_name)
        axes[2].plot(
            history['epoch'],
            history['mean_source_validation_macro_f1'],
            label=result_name,
        )
    axes[0].set(title='Source classification loss', xlabel='Epoch', ylabel='Loss')
    axes[1].set(title='Pairwise source MMD', xlabel='Epoch', ylabel='Penalty')
    axes[2].set(title='Mean source validation macro-F1', xlabel='Epoch', ylabel='Macro-F1')
    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend()
    figure.tight_layout()
    figure.savefig(output_file, dpi=200, bbox_inches='tight')
    return figure

In [ ]:
# Plot and save source-only histories before any target evaluation is possible.
# The figure provides classification and MMD training evidence for the report.

TRAINING_CURVE_FILE = OUTPUT_PATHS['figures_directory'] / 'main_training_curves.png'
plot_training_curves(TRAINING_RESULTS, TRAINING_CURVE_FILE)
print(f'Saved: {TRAINING_CURVE_FILE}')

In [ ]:
# Define a source-selection table for reused ERM and both trained methods.
# Every selected value is mean macro-F1 over the same three source validations.

def create_selection_table(training_results):
    return pd.DataFrame([
        {
            'result_name': result_name,
            'selected_epoch': result['selected_epoch'],
            'mean_source_validation_macro_f1': result['selection_value'],
            'reused_from_task2': result['reused_from_task2'],
            'checkpoint_file': str(result['checkpoint_file']),
        }
        for result_name, result in training_results.items()
    ])

In [ ]:
# Display all source-based selections without loading or inspecting Sketch.
# Confirm ERM is marked as reused while DAN-DG and SAM are Task 3 runs.

display(create_selection_table(TRAINING_RESULTS))

## 8. Source-domain separability and common sharpness

In [ ]:
# Define source-only diagnostics for one fixed model and common validation batch.
# The model is released after computing source metrics, separability, and sharpness.

def evaluate_source_diagnostics(configuration, datasets, fixed_batch):
    result = evaluate_source_model(configuration, datasets, DEVICE)
    diagnostics = configuration['diagnostics']
    separability = evaluate_source_domain_separability(
        result['model'],
        result['validation_loaders'],
        DEVICE,
        seed=configuration['seed'],
        training_ratio=diagnostics['diagnostic_train_ratio'],
        logistic_regression_c=diagnostics['logistic_regression_c'],
    )
    sharpness = calculate_local_sharpness(
        result['model'],
        fixed_batch,
        DEVICE,
        radius=diagnostics['sharpness_radius'],
    )
    source_metrics = result['metrics']
    result['model'].to('cpu')
    del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {
        'source_metrics': source_metrics,
        'source_domain_separability': separability,
        'sharpness': sharpness,
    }

In [ ]:
# Materialize exactly 32 fixed validation examples from each source domain.
# Reuse this identical 96-image tensor batch for every model's sharpness proxy.

FIXED_SHARPNESS_BATCH, FIXED_SHARPNESS_INDICES = create_fixed_validation_batch(
    TASK3_DATASETS['source_validation'],
    examples_per_domain=32,
    seed=6304,
)
save_json(
    FIXED_SHARPNESS_INDICES,
    OUTPUT_PATHS['metrics_directory'] / 'source_sharpness_indices.json',
)

In [ ]:
# Compute the two required diagnostics for ERM, DAN-DG, and SAM using sources only.
# Save these measurements before the experiment lock and target-label evaluation.

SOURCE_DIAGNOSTICS = {
    result_name: evaluate_source_diagnostics(
        configuration,
        TASK3_DATASETS,
        FIXED_SHARPNESS_BATCH,
    )
    for result_name, configuration in MAIN_CONFIGURATIONS.items()
}
save_json(
    SOURCE_DIAGNOSTICS,
    OUTPUT_PATHS['metrics_directory'] / 'source_only_diagnostics.json',
)

In [ ]:
# Define a compact table for mean/worst source metrics and both diagnostics.
# This table can be interpreted safely before any Sketch images are loaded.

def create_source_diagnostic_table(source_diagnostics):
    return pd.DataFrame([
        {
            'result_name': result_name,
            'mean_source_accuracy': result['source_metrics']['mean_accuracy'],
            'mean_source_macro_f1': result['source_metrics']['mean_macro_f1'],
            'worst_source_accuracy': result['source_metrics']['worst_accuracy'],
            'worst_source_macro_f1': result['source_metrics']['worst_macro_f1'],
            'source_domain_separability': result['source_domain_separability']['accuracy'],
            'sharpness_increase': result['sharpness']['sharpness_increase'],
        }
        for result_name, result in source_diagnostics.items()
    ])

In [ ]:
# Display source-only evidence and compare separability with its 33.3% chance level.
# Do not infer final Sketch performance from either diagnostic alone.

display(create_source_diagnostic_table(SOURCE_DIAGNOSTICS))

## 9. Controlled SAM radius study

In [ ]:
# Record student-written expectations before any controlled-study target results.
# Replace every TODO without using Task 2 Sketch results to revise Task 3 choices.

HYPOTHESES = {
    'main_methods': 'TODO: predict ERM, DAN-DG, and SAM trade-offs',
    'sam_radius': 'TODO: predict source, sharpness, and Sketch trends',
    'task2_vs_task3_dan': 'TODO: predict the value of unlabeled target access',
}
save_json(HYPOTHESES, OUTPUT_PATHS['metrics_directory'] / 'task3_hypotheses.json')
display(HYPOTHESES)

In [ ]:
# Define the two extra SAM radii around the fixed main radius of 0.05.
# Unique run names prevent controlled checkpoints from replacing main SAM.

def build_sam_design_configurations():
    return {
        'sam_rho_0p01': load_config('sam', overrides={
            'method': {'run_name': 'sam_rho_0p01', 'rho': 0.01}
        }),
        'sam_rho_0p1': load_config('sam', overrides={
            'method': {'run_name': 'sam_rho_0p1', 'rho': 0.1}
        }),
    }

In [ ]:
# Create and display the two additional fixed SAM configurations.
# The main SAM run supplies rho 0.05 without duplicate training.

DESIGN_CONFIGURATIONS = build_sam_design_configurations()
display(pd.DataFrame([
    {'result_name': name, 'rho': config['method']['rho']}
    for name, config in DESIGN_CONFIGURATIONS.items()
]))

In [ ]:
# Define sequential source-only training for the extra SAM radius settings.
# Every non-radius setting remains identical to the main SAM experiment.

def train_sam_design_study(configurations, datasets, force_retrain=False):
    return {
        result_name: run_and_release(
            configuration,
            datasets,
            force_retrain=force_retrain,
        )
        for result_name, configuration in configurations.items()
    }

In [ ]:
# Train or load both extra SAM runs and show their source-only selections.
# Complete both checkpoints before creating the experiment lock.

DESIGN_TRAINING_RESULTS = train_sam_design_study(
    DESIGN_CONFIGURATIONS,
    TASK3_DATASETS,
    force_retrain=FORCE_RETRAIN,
)
display(create_selection_table(DESIGN_TRAINING_RESULTS))

## 10. Lock all decisions before loading Sketch

In [ ]:
# Combine main and controlled configurations for one immutable experiment lock.
# The lock covers the reused ERM and every newly trained Task 3 checkpoint.

ALL_CONFIGURATIONS = {**MAIN_CONFIGURATIONS, **DESIGN_CONFIGURATIONS}
LOCK_FILE = OUTPUT_PATHS['metrics_directory'] / 'task3_experiment_lock.json'
print('Runs to lock:', list(ALL_CONFIGURATIONS))

In [ ]:
# Hash the hypotheses, configurations, and checkpoints before target access.
# Incomplete hypotheses or any later change prevents final evaluation.

EXPERIMENT_LOCK = create_experiment_lock(ALL_CONFIGURATIONS, LOCK_FILE)
display(pd.DataFrame(EXPERIMENT_LOCK['runs']).T[[
    'method', 'run_name', 'checkpoint_sha256', 'configuration_sha256'
]])

## 11. Final Sketch evaluation

This is the only stage allowed to construct a Sketch dataset. Run it once after every checkpoint, configuration, source diagnostic, and hypothesis is fixed. Do not use these final results to revise Task 3.

In [ ]:
# Keep Sketch access disabled until every preceding experiment decision is locked.
# Change this value to True only when ready for final target analysis.

UNLOCK_SKETCH = False
print('Final Sketch evaluation unlocked:', UNLOCK_SKETCH)

In [ ]:
# Run the protected evaluator only after the explicit final unlock decision.
# It saves aggregate, diagnostic, class-level, failure, and cross-task evidence.

if UNLOCK_SKETCH:
    FINAL_RESULTS = run_final_sketch_evaluation(
        ALL_CONFIGURATIONS,
        TASK3_DATASETS,
        PACS_SOURCE_ROOT,
        LOCK_FILE,
        device=DEVICE,
    )
else:
    print('Sketch evaluation skipped. Lock decisions, then set the flag to True.')

## 12. Final evidence tables

In [ ]:
# Display the complete Task 3 comparison after final Sketch evaluation.
# It includes each source, mean/worst source, Sketch, diagnostics, and ERM delta.

if 'FINAL_RESULTS' in globals():
    display(FINAL_RESULTS['summary'])
else:
    print('Run the locked final Sketch evaluation first.')

In [ ]:
# Define the controlled SAM table from the three fixed final radius results.
# Sort by rho to compare source, sharpness, and Sketch behavior directly.

def create_sam_radius_table(summary_table):
    rho_by_result = {'sam_rho_0p01': 0.01, 'sam': 0.05, 'sam_rho_0p1': 0.1}
    table = summary_table[summary_table['result_name'].isin(rho_by_result)].copy()
    table['rho'] = table['result_name'].map(rho_by_result)
    columns = [
        'rho',
        'mean_source_accuracy',
        'worst_source_accuracy',
        'source_domain_separability',
        'sharpness_increase',
        'sketch_accuracy',
        'sketch_macro_f1',
    ]
    return table.sort_values('rho')[columns].reset_index(drop=True)

In [ ]:
# Display the bounded SAM radius study after all locked target results exist.
# Interpret its trends against the hypothesis written before target access.

if 'FINAL_RESULTS' in globals():
    SAM_RADIUS_TABLE = create_sam_radius_table(FINAL_RESULTS['summary'])
    display(SAM_RADIUS_TABLE)
else:
    print('Run the locked final Sketch evaluation first.')

In [ ]:
# Display the largest class improvement and degradation relative to shared ERM.
# Detailed class tables and confusion figures are saved under task3/results.

if 'FINAL_RESULTS' in globals():
    display(pd.DataFrame(FINAL_RESULTS['class_analysis']).T)
else:
    print('Run the locked final Sketch evaluation first.')

In [ ]:
# Display target-aware Task 2 DAN beside target-free Task 3 DAN-DG when available.
# This comparison is intentionally performed only after Task 3 has been locked.

if 'FINAL_RESULTS' in globals() and FINAL_RESULTS['task2_comparison'] is not None:
    display(FINAL_RESULTS['task2_comparison'])
else:
    print('Run final evaluations for Tasks 2 and 3 to create this comparison.')

In [ ]:
# Define a loader for detailed per-class Sketch changes saved by each main method.
# Separate table loading keeps the notebook results concise but fully inspectable.

def load_class_change_tables(result_names, metrics_directory):
    return {
        result_name: pd.read_csv(
            metrics_directory / f'{result_name}_class_changes.csv'
        )
        for result_name in result_names
    }

In [ ]:
# Display detailed class-level changes for ERM, DAN-DG, and SAM.
# Use failure CSV files and confusion figures for selected image inspection.

if 'FINAL_RESULTS' in globals():
    CLASS_CHANGE_TABLES = load_class_change_tables(
        MAIN_CONFIGURATIONS.keys(),
        OUTPUT_PATHS['metrics_directory'],
    )
    for result_name, table in CLASS_CHANGE_TABLES.items():
        print(result_name)
        display(table)
else:
    print('Run the locked final Sketch evaluation first.')